# Faza 4 — Scalare la Modele Mai Mari + FP8 Native
## ViT-Tiny / ViT-Small / ViT-Base pe ImageNet-1k · NVIDIA A100
### Tudose Alexandru · Lucrare de Licență 2026

**Ce rulează acest notebook:**
1. FP32 / FP16 / INT8-pt / INT8-pc pe ViT-Tiny, ViT-Small, ViT-Base
2. FP8 E4M3FN (per-tensor + per-channel) pe toate trei modelele
3. Comparație cross-model cu ploturi

## 0. Verificare GPU

In [ ]:
import torch
print(torch.cuda.get_device_name(0))
print(f"CUDA: {torch.version.cuda}")
print(f"PyTorch: {torch.__version__}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
cap = torch.cuda.get_device_capability()
print(f"Compute capability: {cap[0]}.{cap[1]}")
if cap[0] >= 9:
    print("→ Hopper: hardware FP8 GEMM disponibil")
elif cap[0] >= 8:
    print("→ Ampere: FP8 dtype suportat, fara hardware FP8 GEMM")
print(f"FP8 (float8_e4m3fn) available: {hasattr(torch, 'float8_e4m3fn')}")

## 1. Setup — Drive, Repo, Dependente

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!git clone https://github.com/Chewdoughsay/ViT-FP8-experiments.git
%cd ViT-FP8-experiments

In [ ]:
%pip install -q timm datasets huggingface_hub

## 2. Copiere ImageNet de pe Drive

In [ ]:
import shutil, os

os.makedirs('data/imagenet-1k', exist_ok=True)
shutil.copytree(
    '/content/drive/MyDrive/imagenet-1k',
    'data/imagenet-1k',
    dirs_exist_ok=True
)
print("Dataset copiat OK")
!ls data/imagenet-1k/

## 3. Sanity check

In [ ]:
import sys
sys.path.insert(0, '/content/ViT-FP8-experiments')

import timm, timm.data, torch
from src.models.quantized_linear import quantize_model_selective

model = timm.create_model('vit_tiny_patch16_224', pretrained=True).cuda().eval()
model, stats = quantize_model_selective(model, verbose=False)
x = torch.randn(4, 3, 224, 224).cuda()
with torch.no_grad():
    out = model(x)
print(f"Forward pass OK — {out.shape}")
del model

## 4. Evaluare FP32 / FP16 / INT8-pt / INT8-pc

In [ ]:
!python scripts/phase4/evaluate_model.py \
    --model vit_tiny_patch16_224 \
    --batch-size 128

In [ ]:
!python scripts/phase4/evaluate_model.py \
    --model vit_small_patch16_224 \
    --batch-size 128

In [ ]:
!python scripts/phase4/evaluate_model.py \
    --model vit_base_patch16_224 \
    --batch-size 64

In [ ]:
import shutil
shutil.copytree(
    '/content/ViT-FP8-experiments/results/Phase4',
    '/content/drive/MyDrive/vit_experiments/results/Phase4',
    dirs_exist_ok=True
)
print("Rezultate INT8 salvate pe Drive!")

## 5. FP8 Native (float8_e4m3fn)

Rulează FP32 + INT8-pt + INT8-pc + **FP8-pt + FP8-pc** pe fiecare model.

Pe A100 (Ampere, compute capability 8.0): `float8_e4m3fn` este suportat ca dtype PyTorch,
dar fara hardware FP8 GEMM nativ (necesita H100 Hopper). Comparatia de acuratete este valida;
latenta FP8 nu reflecta potentialul hardware real.

In [ ]:
!python scripts/phase4/evaluate_fp8_native.py \
    --model vit_tiny_patch16_224 \
    --batch-size 128

In [ ]:
!python scripts/phase4/evaluate_fp8_native.py \
    --model vit_small_patch16_224 \
    --batch-size 128

In [ ]:
!python scripts/phase4/evaluate_fp8_native.py \
    --model vit_base_patch16_224 \
    --batch-size 64

In [ ]:
import shutil
shutil.copytree(
    '/content/ViT-FP8-experiments/results/Phase4',
    '/content/drive/MyDrive/vit_experiments/results/Phase4',
    dirs_exist_ok=True
)
print("Rezultate FP8 salvate pe Drive!")

## 6. Comparatie cross-model

In [ ]:
!python scripts/phase4/compare_cross_model.py

In [ ]:
from IPython.display import Image as IPImage, display
from pathlib import Path

for p in sorted(Path('results/Phase4/cross_model_comparison').glob('*.png')):
    print(f'\n--- {p.name} ---')
    display(IPImage(str(p)))

## 7. Salvare finala pe Drive

In [ ]:
import shutil
shutil.copytree(
    '/content/ViT-FP8-experiments/results/Phase4',
    '/content/drive/MyDrive/vit_experiments/results/Phase4',
    dirs_exist_ok=True
)
print("Toate rezultatele salvate pe Drive!")